# 이상치 탐지(IQR/Z-score)와 통계적 유의성 검정
- IQR(사분위범위) 방식과 Z-score 방식, 두 가지로 이상치를 탐지하고 결과를 비교
- scipy의 t-검정으로 "두 카테고리의 평균 가격 차이가 통계적으로 유의미한가?"를 직접 검증

### 💡 강의 포인트
- boxplot으로 "눈으로 보는" 이상치(예제06)와, 이번에 "수치로 판별하는" 이상치를 연결해서 설명하면 학생들이 흐름을 이어서 이해하기 좋음.
- t-검정 결과에서 가장 중요한 건 **p-value**라는 것, 그리고 "p-value < 0.05면 통계적으로 유의미하다고 본다"는 일반적인 기준(반드시 진리는 아니라는 점도 함께 언급)을 짚어주기.
- `scipy`는 별도 설치가 필요합니다: `uv pip install scipy`

> ⚠️ 이번 예제는 Chapter02에서 가장 난이도가 높은 예제입니다. 통계 개념(사분위수, 표준편차, 가설검정)에 대한 사전 설명을 5~10분 정도 먼저 하고 들어가는 걸 추천합니다.

In [ ]:
import os
from dotenv import load_dotenv
import oracledb
import pandas as pd

load_dotenv() 

USER = os.getenv("ORACLE_USER")
PASSWORD = os.getenv("ORACLE_PASSWORD")
DSN = os.getenv("ORACLE_DSN")


In [ ]:
with oracledb.connect(user=USER, password=PASSWORD, dsn=DSN) as conn:
    df = pd.read_sql("SELECT * FROM DELIVERY_ORDERS", conn)


df.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rc('font', family='Malgun Gothic')  
mpl.rc('axes', unicode_minus=False)


In [ ]:
import numpy as np

Q1 = df["PRICE"].quantile(0.25)
Q3 = df["PRICE"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_outliers = df[(df["PRICE"] < lower_bound) | (df["PRICE"] > upper_bound)]
print(f"IQR 범위: {lower_bound:.0f} ~ {upper_bound:.0f}")
print(f"IQR 기준 이상치 {len(iqr_outliers)}건")
print(iqr_outliers[["MENU_NAME", "PRICE"]])

In [ ]:
mean_price = df["PRICE"].mean()
std_price = df["PRICE"].std()
df["Z_SCORE"] = (df["PRICE"] - mean_price) / std_price

z_outliers = df[df["Z_SCORE"].abs() > 2]
print(f"Z-score 기준(|Z|>2) 이상치 {len(z_outliers)}건")
print(z_outliers[["MENU_NAME", "PRICE", "Z_SCORE"]])

In [ ]:
from scipy import stats

group_a = df[df["CATEGORY"] == "치킨"]["PRICE"]
group_b = df[df["CATEGORY"] == "중식"]["PRICE"]

t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)

print(f"치킨 평균가격: {group_a.mean():.0f}원 (n={len(group_a)})")
print(f"중식 평균가격: {group_b.mean():.0f}원 (n={len(group_b)})")
print(f"\nt-통계량: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("→ 통계적으로 유의미한 차이가 있다고 볼 수 있음 (p < 0.05)")
else:
    print("→ 통계적으로 유의미한 차이라고 보기 어려움 (p >= 0.05)")

In [ ]:
import seaborn as sns

plt.figure(figsize=(6, 5))
sns.boxplot(data=df[df["CATEGORY"].isin(["치킨", "중식"])], x="CATEGORY", y="PRICE")
plt.title("치킨 vs 중식 가격 분포 비교")
plt.show()

### ❓ 생각해볼 질문
1. IQR 방식과 Z-score 방식으로 찾은 이상치 개수가 다르다면, 왜 그럴까? (두 방식의 기준이 다름을 정리)
2. 표본 개수(n)가 매우 적을 때 t-검정 결과(p-value)를 얼마나 신뢰할 수 있을까? 실습 데이터를 늘려서 다시 검정해보면 결과가 어떻게 바뀔까?